<a href="https://colab.research.google.com/github/natasya783/Perhitungan-Target-Servis/blob/main/Riwayat%20Servis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 4. PERHITUNGAN TARGET SERVIS (Bobot 15%)
# =====================================================================
INTERVAL_SERVIS = {
    "Ganti Oli": 2000,
    "Oli Gardan": 5000,
    "Tune Up": 8000,
    "Kampas Rem": 10000,
    "Ban": 20000,
}

def hitung_target_servis(jenis_servis, km_saat_servis):
    """Menghitung target kilometer berikutnya berdasarkan jenis servis."""
    return km_saat_servis + INTERVAL_SERVIS[jenis_servis]

def jadwal_servis_berikutnya(user_id):
    """Menampilkan target terakhir untuk setiap jenis servis pada kendaraan."""
    kendaraan = pilih_kendaraan(user_id)
    if not kendaraan:
        return
    id_kendaraan, km_sekarang = kendaraan[0], kendaraan[5]
    with koneksi_db() as conn:
        cur = conn.execute("""
            SELECT r.jenis_servis, r.tanggal_servis, r.km_saat_servis,
                   r.km_target_berikutnya
            FROM riwayat_servis r
            JOIN (
                SELECT jenis_servis, MAX(id) AS id_terakhir
                FROM riwayat_servis
                WHERE id_kendaraan = ?
                GROUP BY jenis_servis
            ) terakhir ON r.id = terakhir.id_terakhir
            ORDER BY r.jenis_servis
        """, (id_kendaraan,))
        data = cur.fetchall()

    print("\n=== JADWAL SERVIS BERIKUTNYA ===")
    if not data:
        print("Belum ada riwayat servis untuk kendaraan ini.")
        return
    print(f"KM kendaraan saat ini: {km_sekarang:,} Km")
    for jenis, tanggal, km_servis, target in data:
        print(f"\n{jenis} | servis terakhir: {tanggal} pada {km_servis:,} Km")
        print(f"Target berikutnya: {target:,} Km")